In [ ]:
# GPT-2 FineWeb pretraining using the reviewed repository implementation
!nvidia-smi
import torch

num_gpus = torch.cuda.device_count()
gpu_names = [torch.cuda.get_device_name(index) for index in range(num_gpus)]
print(f"GPU count: {num_gpus}")
for index, name in enumerate(gpu_names):
    print(f"GPU {index}: {name}")
if num_gpus != 2 or any("T4" not in name for name in gpu_names):
    raise RuntimeError(
        "This training requires two Tesla T4 GPUs. In Kaggle, select GPU T4 x2; "
        f"the allocated hardware is {gpu_names}."
    )


In [ ]:
# Install exactly the branch that contains this recipe.
!pip install -q uv
!git clone --depth 1 --branch spirlness/feat/gpt2-fineweb-training https://github.com/spirlness/Automodel.git Automodel
%cd Automodel
!uv sync --locked --group dev --extra fa --inexact

# Store the token only in Kaggle Secrets under the name HF_TOKEN. Never print it.
from kaggle_secrets import UserSecretsClient
import os

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("Loaded HF_TOKEN from Kaggle Secrets.")


In [ ]:
# First verify download, tokenization, and binary writing on a bounded sample.
!uv run python projects/gpt2_fineweb_500m/tools/nanogpt_data_processor.py   --dataset HuggingFaceFW/fineweb   --set-name sample-10BT   --output-dir /kaggle/working/fineweb_smoke   --max-tokens 1M   --chunk-size 16   --prefetch 4   --num-workers 2


In [ ]:
# Exercise two-rank FSDP, FP16, ordinary CE, backward, and optimizer update.
# T4 supports at most 64 KiB shared memory per block, so use logits-based CE instead
# of the incompatible fused Triton CE. This runs one full-size global training step.
!PYTORCH_ALLOC_CONF=expandable_segments:True uv run automodel projects/gpt2_fineweb_500m/config/gpt2_fineweb_500m.yaml   --nproc-per-node 2   --dataset.file_pattern=/kaggle/working/fineweb_smoke_max_tokens_1M/dataset.bin   --step_scheduler.global_batch_size=32   --step_scheduler.local_batch_size=4   --step_scheduler.max_steps=1   --step_scheduler.ckpt_every_steps=1   --step_scheduler.val_every_steps=1000000   --step_scheduler.save_checkpoint_every_epoch=false   --checkpoint.enabled=false   --loss_fn._target_=nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy   --model.torch_dtype=float16   --distributed.mp_policy.param_dtype=torch.float16   --distributed.mp_policy.output_dtype=torch.float16


In [ ]:
# Smoke test passed. Keep this False until its logs confirm both preprocessing
# and one optimizer update completed successfully. Then set it to True and run this cell.
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    get_ipython().system(
        "PYTORCH_ALLOC_CONF=expandable_segments:True uv run python "
        "projects/gpt2_fineweb_500m/tools/nanogpt_data_processor.py "
        "--dataset HuggingFaceFW/fineweb --set-name sample-10BT "
        "--output-dir /kaggle/working/fineweb_1B --max-tokens 1B"
    )
    get_ipython().system(
        "PYTORCH_ALLOC_CONF=expandable_segments:True uv run automodel projects/gpt2_fineweb_500m/config/gpt2_fineweb_500m.yaml "
        "--nproc-per-node 2 "
        "--dataset.file_pattern=/kaggle/working/fineweb_1B_max_tokens_1B/dataset.bin "
        "--step_scheduler.global_batch_size=32 "
        "--step_scheduler.local_batch_size=4 "
        "--step_scheduler.max_steps=30517 "
        "--step_scheduler.ckpt_every_steps=10000 "
        "--step_scheduler.save_checkpoint_every_epoch=false "
        "--checkpoint.checkpoint_dir=/kaggle/working/checkpoints "
        "--checkpoint.max_recent_checkpoints=3 "
        "--loss_fn._target_=nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy "
        "--model.torch_dtype=float16 "
        "--distributed.mp_policy.param_dtype=torch.float16 "
        "--distributed.mp_policy.output_dtype=torch.float16"
    )
else:
    print("Full 1B-token training is disabled until the smoke test passes.")
